# Construcción del Patch Store por Sitio

Convierte los archivos MCMIPF horarios completos (C×16×256×256) en arrays recortados
centrados en cada sitio de interés. Esto reduce drásticamente el I/O durante el entrenamiento.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd

In [2]:
def extract_patch(frame_c_hw: np.ndarray, center_rc: tuple[int,int], patch: int) -> np.ndarray:
    r0, c0 = center_rc
    half = patch // 2
    r1, r2 = r0 - half, r0 + half
    c1, c2 = c0 - half, c0 + half
    return frame_c_hw[:, r1:r2, c1:c2]  # (C,P,P)

def hour_path_for_timestamp(mcmipf_root: Path, t: pd.Timestamp) -> Path:
    ymd = t.strftime("%Y%m%d")
    hh  = t.strftime("%H")
    year  = t.strftime("%Y")
    month = t.strftime("%m")
    fname = f"{ymd}_{hh}_MCMIPF.npz"
    return mcmipf_root / year / month / fname

def build_patch_store_for_site(
    site: str,
    dataset_meta_path: Path,
    manifests_dir: Path,
    out_root: Path,
    patch: int,
    dtype: str = "float16",
) -> None:
    with open(dataset_meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    mcmipf_root = Path(meta["mcmipf_root"])
    center = (int(meta["site_center_pix"]["row"]), int(meta["site_center_pix"]["col"]))
    grid_size = int(meta["grid_size"])

    assert patch % 2 == 0
    half = patch // 2
    assert 0 <= center[0]-half and center[0]+half <= grid_size
    assert 0 <= center[1]-half and center[1]+half <= grid_size

    # Juntar todos los timestamps de historia para cubrir lo que de verdad se usa
    ts_all: set[pd.Timestamp] = set()
    for split in ["train", "val", "test"]:
        man = pd.read_parquet(manifests_dir / f"manifest_{split}.parquet")
        for x in man["history_ts"]:
            if isinstance(x, str):
                x = json.loads(x)
            for ts_str in x:
                ts_all.add(pd.to_datetime(ts_str, utc=True))

    # Convertir a “hour keys” (ymd,hh)
    hour_keys: set[tuple[str,str]] = set((t.strftime("%Y%m%d"), t.strftime("%H")) for t in ts_all)

    print(f"[{site}] unique history timestamps: {len(ts_all)}")
    print(f"[{site}] unique hour files needed: {len(hour_keys)}")

    for (ymd, hh) in sorted(hour_keys):
        # construir un timestamp representativo para resolver ruta
        t = pd.Timestamp(f"{ymd} {hh}:00:00", tz="UTC")
        src = hour_path_for_timestamp(mcmipf_root, t)

        if not src.exists():
            # si falta, no revientes: el manifest debería ya filtrar cobertura,
            # pero por si acaso.
            continue

        year = t.strftime("%Y")
        month = t.strftime("%m")
        out_dir = out_root / site / f"P{patch}" / year / month
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{ymd}_{hh}_patch.npz"

        if out_path.exists():
            continue

        with np.load(src) as data:
            arr = data["mcmipf"].astype(np.float32)  # (6,16,256,256)

        # Extraer patch para las 6 ranuras
        patches = []
        for s in range(arr.shape[0]):
            frame = arr[s]  # (16,256,256)
            p = extract_patch(frame, center, patch)  # (16,P,P)
            p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
            patches.append(p)

        patch_arr = np.stack(patches, axis=0)  # (6,16,P,P)

        if dtype == "float16":
            patch_arr = patch_arr.astype(np.float16)
        else:
            patch_arr = patch_arr.astype(np.float32)

        np.savez(out_path, patch=patch_arr)

    print(f"[{site}] patch store built at: {out_root/site/f'P{patch}'}")

if __name__ == "__main__":
    PROJECT_ROOT = Path("..").resolve()
    OUT_ROOT = PROJECT_ROOT / "data" / "patches_v1"
    DATASET_ROOT = PROJECT_ROOT / "data" / "datasets" / "manifest_v1"

    for site in ["uniandes", "elpaso"]:
        site_dir = DATASET_ROOT / site
        build_patch_store_for_site(
            site=site,
            dataset_meta_path=site_dir / "dataset_meta.json",
            manifests_dir=site_dir,
            out_root=OUT_ROOT,
            patch=16,          
            dtype="float16",
        )

[uniandes] unique history timestamps: 81064
[uniandes] unique hour files needed: 13512
[uniandes] patch store built at: /srv/projects/Proyecto_e_ladino/data/patches_v1/uniandes/P16
[elpaso] unique history timestamps: 104580
[elpaso] unique hour files needed: 17430
[elpaso] patch store built at: /srv/projects/Proyecto_e_ladino/data/patches_v1/elpaso/P16
